# Order Latency Data

To obtain more realistic backtesting results, accounting for latencies is crucial. Therefore, it's important to collect both feed data and order data with timestamps to measure your order latency. The best approach is to gather your own order latencies. You can collect order latency based on your live trading or by regularly submitting orders at a price that cannot be filled and then canceling them for recording purposes. However, if you don't have access to them or want to establish a target, you will need to artificially generate order latency. You can model this latency based on factors such as feed latency, trade volume, and the number of events. In this guide, we will demonstrate a simple method to generate order latency from feed latency using a multiplier and offset for adjustment.

## Runnable Tardis Test

This notebook executes the corresponding experiment through the shared
`tutorial_reproduction` runner. It uses existing Tardis files only and
does not require a Tardis API key or download data.

Defaults:

- amdserver: `/home/molly/data/tardis/binance-futures`, `2025-08-01`
- Mac: `~/Documents/tardis`, `2025-01-01`
- Window: `300` seconds

Optional environment overrides:

- `HFTBACKTEST_TARDIS_ROOT`
- `HFTBACKTEST_TARDIS_DATE`
- `HFTBACKTEST_NOTEBOOK_SECONDS`
- `HFTBACKTEST_NOTEBOOK_OUTPUT`

Active experiment: `Order Latency Data.ipynb` (`order_latency_data`).

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
search_roots = []
for base in (cwd, *cwd.parents):
    search_roots.extend((base, base / 'examples'))
examples_root = next(
    path for path in search_roots
    if (path / 'tutorial_reproduction').is_dir()
)
if str(examples_root) not in sys.path:
    sys.path.insert(0, str(examples_root))

from tutorial_reproduction.notebook_support import (
    context_dict,
    notebook_context,
    run_notebook_experiment,
)

In [ ]:
context = notebook_context()
context_dict(context)

In [ ]:
manifest = run_notebook_experiment('order_latency_data', context)
manifest['result']

In [ ]:
assert manifest['result']['status'] != 'failed'
print('notebook:', manifest['notebook'])
print('status:', manifest['result']['status'])
print('output:', context.output_root)

## Original Tutorial Reference

The original tutorial narrative and code are retained below for comparison.
Original code cells are rendered as non-executing references so that
`Run All` remains reproducible with the configured Tardis dataset.

First, loads the feed data.

For easy manipulation, converts it into a DataFrame.

Selects only the events that have both a valid exchange timestamp and a valid local timestamp to get feed latency.

Reduces the number of rows by resampling to approximately 1-second intervals.

Converts back to the structured NumPy array.

Generates order latency. Order latency consists of two components: the latency until the order request reaches the exchange's matching engine and the latency until the response arrives backto the localy. Order latency is not the same as feed latency and does not need to be proportional to feed latency. However, for simplicity, we model order latency to be proportional to feed latency using a multiplier and offset.

Checks if latency has invalid negative values.

Here, we wrap the entire process into a method with `njit` for increased speed.